# ALGO XGBoost next-day direction model

This notebook trains an XGBoost classifier to predict whether ALGO's next-day return is positive. Features use today's prices of every other asset plus ALGO EWMA features. The split is time-based: train on days before 400, test on days 400-499.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from xgboost import XGBClassifier

TRAIN_END = 400
TARGET_ASSET = "ALGO"
EWMA_SPANS = [5, 10, 20, 60]

prices = pd.read_csv("prices.txt", sep=r"\s+")
prices.head()

## Build features and target

For day `t`, the target is whether ALGO's return from day `t` to day `t + 1` is greater than zero. The model sees prices and EWMA features from day `t`, not day `t + 1`.

In [ ]:
def build_algo_features(prices: pd.DataFrame, target_asset: str = TARGET_ASSET) -> pd.DataFrame:
    features = pd.DataFrame(index=prices.index)

    other_assets = [asset for asset in prices.columns if asset != target_asset]
    for asset in other_assets:
        features[f"price_{asset}"] = prices[asset]

    algo_price = prices[target_asset]
    algo_return = algo_price.pct_change()

    features["algo_price"] = algo_price
    features["algo_return_1"] = algo_return
    features["algo_return_5"] = algo_price.pct_change(5)
    features["algo_vol_20"] = algo_return.rolling(20).std()

    for span in EWMA_SPANS:
        ewma = algo_price.ewm(span=span, adjust=False).mean()
        features[f"algo_ewma_{span}"] = ewma
        features[f"algo_price_to_ewma_{span}"] = algo_price / ewma - 1.0
        features[f"algo_ewma_return_{span}"] = ewma.pct_change()

    next_return = algo_price.pct_change().shift(-1)
    features["algo_return_next"] = next_return
    features["target_up"] = (next_return > 0).astype(int)

    return features.dropna()

data = build_algo_features(prices)

feature_cols = [col for col in data.columns if col not in ["algo_return_next", "target_up"]]
train = data[data.index < TRAIN_END]
test = data[data.index >= TRAIN_END]

X_train = train[feature_cols]
y_train = train["target_up"]
X_test = test[feature_cols]
y_test = test["target_up"]

print(f"Feature count: {len(feature_cols)}")
print(f"Train rows: {len(train)}")
print(f"Test rows: {len(test)}")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test positive rate: {y_test.mean():.3f}")

## Train XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=5.0,
    reg_alpha=0.5,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=20260710,
    n_jobs=1,
)

model.fit(X_train, y_train)

train_prob = model.predict_proba(X_train)[:, 1]
test_prob = model.predict_proba(X_test)[:, 1]

train_pred = (train_prob > 0.5).astype(int)
test_pred = (test_prob > 0.5).astype(int)

results = pd.Series({
    "train_auc": roc_auc_score(y_train, train_prob),
    "test_auc": roc_auc_score(y_test, test_prob),
    "train_accuracy": accuracy_score(y_train, train_pred),
    "test_accuracy": accuracy_score(y_test, test_pred),
})

results

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance.head(20)

In [ ]:
ax = importance.head(20).sort_values().plot(kind="barh", figsize=(9, 6))
ax.set_title("Top 20 XGBoost feature importances")
ax.set_xlabel("Importance")
plt.tight_layout()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Test AUC = {roc_auc_score(y_test, test_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.5)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ALGO next-day direction ROC curve")
plt.legend()
plt.tight_layout()

In [ ]:
predictions = pd.DataFrame({
    "actual_up": y_test,
    "prob_up": test_prob,
    "pred_up": test_pred,
    "next_return": test["algo_return_next"],
})

predictions.head(20)

## Current run summary

With the parameters above, the terminal run gave approximately:

- Train AUC: 0.821
- Test AUC: 0.540
- Train accuracy: 0.763
- Test accuracy: 0.485

The AUC is the cleaner metric here because the default 0.5 threshold is poorly calibrated: the model ranks some up days better than random, but its raw probabilities are not yet a good standalone trading rule.